#  Multi-Source Data Scraper & Trust Scoring System



##  README — Project Documentation

# Multi-Source Data Scraper & Trust Scoring System

A Python pipeline that scrapes structured content from **blog posts**, **YouTube videos**, and **PubMed articles**, then evaluates each source's reliability using a weighted trust scoring algorithm.

---

## Project Structure

```
project/
├── main.py                  # Entry point – runs full pipeline
├── scraper/
│   ├── blog_scraper.py      # Blog post scraper (requests + BeautifulSoup)
│   ├── youtube_scraper.py   # YouTube metadata + transcript scraper
│   └── pubmed_scraper.py    # PubMed article scraper (NCBI E-utilities API)
├── scoring/
│   └── trust_score.py       # Trust scoring algorithm
├── utils/
│   ├── tagging.py           # Automatic topic tagging (TF-IDF + domain map)
│   └── chunking.py          # Text chunking utility
└── output/
    ├── blogs.json
    ├── youtube.json
    ├── pubmed.json
    └── scraped_data.json    # Combined dataset (all 6 sources)
```

---

## Tools & Libraries

| Library | Purpose |
|---|---|
| `requests` | HTTP requests for blogs and PubMed |
| `beautifulsoup4` | HTML parsing for blog and YouTube pages |
| `langdetect` | Automatic language detection |
| `scikit-learn` | TF-IDF keyword extraction for topic tagging |
| `youtube-transcript-api` | Fetch YouTube video transcripts/captions |
| `xml.etree.ElementTree` | Parse NCBI PubMed XML responses |

---

## Scraping Approach

### Blogs (`blog_scraper.py`)
- Sends HTTP GET with a browser-like User-Agent header
- Extracts author from JSON-LD schema, `<meta>` tags, or `.author` CSS selectors
- Extracts publication date from `<time>` elements and `article:published_time` meta
- Strips navigation, ads, scripts, and footers before extracting article body
- Detects language using `langdetect` on the first 500 characters

### YouTube (`youtube_scraper.py`)
- Fetches the page HTML and extracts channel name, upload date, and description from meta tags and the embedded `ytInitialData` JSON blob
- Uses `youtube-transcript-api` to download captions where available
- Falls back gracefully if transcripts are unavailable

### PubMed (`pubmed_scraper.py`)
- Uses the NCBI E-utilities REST API (`efetch.fcgi`) with `rettype=xml`
- Parses the XML response to extract title, authors, journal, abstract, and publication year
- Retrieves MeSH (Medical Subject Headings) terms as primary topic tags
- Estimates citation count via NCBI ELink

---

## Trust Score Design

```
Trust Score = w1·author_credibility
            + w2·citation_count
            + w3·domain_authority
            + w4·recency
            + w5·medical_disclaimer_presence
            - penalties
```

| Component | Weight | Description |
|---|---|---|
| author_credibility | 0.25 | Known org affiliation, multi-author average, fallback 0.2 for unknown |
| citation_count | 0.20 | Log-normalised; PubMed citations up to 1000 → 1.0 |
| domain_authority | 0.25 | Seeded DA lookup (0–100 scale); unknown domains default to 35 |
| recency | 0.20 | Exponential decay with 2-year half-life |
| medical_disclaimer | 0.10 | Binary; PubMed always scores 1.0 |

### Penalties (subtracted after weighting)
| Trigger | Penalty |
|---|---|
| Fake/generic author name | −0.10 |
| Blog domain DA < 30 (SEO spam) | −0.12 |
| Health keywords without disclaimer | −0.08 |
| Spam signal keywords in content | −0.05 per signal (max −0.20) |
| Keyword stuffing (token density > 3%) | −0.05 |

Final score is clamped to **[0.0, 1.0]**.

---

## Edge Cases

| Scenario | Handling |
|---|---|
| Missing author | Scored as 0.2 (not zero) to avoid over-penalising anonymous technical content |
| Missing publish date | Recency score defaults to 0.3 (moderate penalty) |
| Transcript unavailable | Returns `[Transcript unavailable]` placeholder; no crash |
| Multiple authors | Author credibility is the average of individual scores |
| Non-English content | `langdetect` auto-detects; stored in `language` field |
| Long articles | `chunk_text()` splits by paragraph then sentence; hard splits at 800 chars |
| Very short chunks | Merged into the previous chunk if below 100 characters |

---

## Abuse Prevention

- **Fake authors**: Names like `admin`, `staff`, `webmaster`, or single-token strings trigger a −0.10 penalty
- **SEO spam blogs**: Domains with DA < 30 receive a −0.12 penalty
- **Misleading medical content**: Health keywords without a medical disclaimer incur −0.08
- **Spam signals**: Phrases like "miracle cure" or "buy now" penalise each occurrence by −0.05
- **Keyword stuffing**: If the most frequent content word exceeds 3% of all tokens, a −0.05 penalty is applied
- **Outdated information**: Exponential recency decay ensures old content cannot achieve high scores

---

## How to Run

### 1. Install dependencies
```bash
pip install requests beautifulsoup4 langdetect scikit-learn youtube-transcript-api
```

### 2. Run the pipeline
```bash
python main.py
```

Output files are written to `output/`.

### 3. Use individual scrapers
```python
from scraper.blog_scraper import scrape_blog
result = scrape_blog("https://example.com/article")
```

---

## Limitations

- Blog scrapers rely on common HTML conventions; highly dynamic or JavaScript-rendered sites may require Playwright/Selenium
- YouTube transcript availability depends on the video having auto-generated or manual captions
- Domain authority values are a seeded static lookup; a live DA API (e.g., Moz, Ahrefs) would give more accurate scores
- PubMed citation counts are approximate (based on NCBI ELink, not citation databases like Scopus)
- `langdetect` is probabilistic and can misidentify language on very short texts


## Step 1 — Install Dependencies

In [ ]:
!pip install requests beautifulsoup4 langdetect scikit-learn youtube-transcript-api -q
print(" All dependencies installed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 44.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 22.5 MB/s eta 0:00:00
 All dependencies installed


## Step 2 — Imports

In [ ]:
import os, re, json, math, requests
import xml.etree.ElementTree as ET
from datetime import datetime
from collections import Counter
from bs4 import BeautifulSoup

try:
    import langdetect
    LANGDETECT_OK = True
except ImportError:
    LANGDETECT_OK = False

print(" Imports complete")


 Imports complete


##  Step 3 — Utility: Text Chunker

In [ ]:
MAX_CHUNK_CHARS = 800
MIN_CHUNK_CHARS = 100

def chunk_text(text, max_chars=MAX_CHUNK_CHARS):
    """Split text into chunks of ~max_chars characters."""
    if not text or not text.strip():
        return []
    text = re.sub(r"\r\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    paragraphs = [p.strip() for p in re.split(r"\n\n+", text) if p.strip()]
    chunks, buffer = [], ""
    for para in paragraphs:
        if len(para) > max_chars:
            if buffer:
                chunks.append(buffer.strip()); buffer = ""
            chunks.extend(_split_sentences(para, max_chars))
        else:
            if len(buffer) + len(para) + 2 <= max_chars:
                buffer = (buffer + "\n\n" + para).strip() if buffer else para
            else:
                if buffer: chunks.append(buffer.strip())
                buffer = para
    if buffer: chunks.append(buffer.strip())
    return _merge_small(chunks)

def _split_sentences(text, max_chars):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, buffer = [], ""
    for s in sentences:
        if len(buffer) + len(s) + 1 <= max_chars:
            buffer = (buffer + " " + s).strip()
        else:
            if buffer: chunks.append(buffer)
            buffer = s if len(s) <= max_chars else (chunks.extend([s[i:i+max_chars] for i in range(0,len(s),max_chars)][:-1]) or s[-max_chars:])
    if buffer: chunks.append(buffer)
    return chunks

def _merge_small(chunks):
    if not chunks: return chunks
    merged = [chunks[0]]
    for c in chunks[1:]:
        if len(c) < MIN_CHUNK_CHARS and merged:
            merged[-1] += "\n\n" + c
        else:
            merged.append(c)
    return merged

print(" Chunker ready")


 Chunker ready


##  Step 4 — Utility: Auto Topic Tagger

In [ ]:
DOMAIN_TAG_MAP = {
    "machine learning":            ["machine learning", "ML", "supervised", "unsupervised"],
    "deep learning":               ["deep learning", "neural network", "transformer", "BERT"],
    "artificial intelligence":     ["artificial intelligence", "AI"],
    "data scraping":               ["web scraping", "data scraping", "crawling", "BeautifulSoup", "scrapy"],
    "natural language processing": ["nlp", "natural language", "text mining", "tokeniz"],
    "healthcare":                  ["health", "medical", "clinical", "patient", "diagnosis", "EHR"],
    "data science":                ["data science", "analytics", "pandas", "dataframe"],
    "python":                      ["python", "pip", ".py", "beautifulsoup"],
    "education":                   ["course", "tutorial", "teach", "student", "learn"],
    "research":                    ["research", "study", "systematic review", "meta-analysis"],
    "computer vision":             ["image recognition", "computer vision", "cnn", "convolutional"],
    "cloud computing":             ["aws", "azure", "gcp", "cloud", "docker", "kubernetes"],
}

STOP_WORDS = {
    "the","a","an","and","or","but","in","on","at","to","for","of","with","by",
    "from","is","are","was","were","be","been","have","has","had","do","does",
    "did","will","would","could","should","this","that","these","those","it",
    "we","you","he","she","they","not","also","more","most","some","any","all",
    "very","just","what","which","who","how","when","where","there","about","into",
}

def auto_tag(text, max_tags=8):
    if not text or not text.strip():
        return []
    text_lower = text.lower()
    tags = []
    for domain, keywords in DOMAIN_TAG_MAP.items():
        if any(kw.lower() in text_lower for kw in keywords):
            tags.append(domain.title())
    remaining = max_tags - len(tags)
    if remaining > 0:
        for kw in _freq_keywords(text, remaining * 2):
            if kw.title() not in tags:
                tags.append(kw.title())
            if len(tags) >= max_tags:
                break
    seen, result = set(), []
    for t in tags:
        if t.lower() not in seen:
            seen.add(t.lower()); result.append(t)
    return result[:max_tags]

def _freq_keywords(text, top_n):
    tokens = re.findall(r"\b[a-zA-Z]{4,}\b", text.lower())
    filtered = [t for t in tokens if t not in STOP_WORDS]
    return [w for w, _ in Counter(filtered).most_common(top_n)]

print(" Tagger ready")


 Tagger ready


##  Step 5 — Trust Score Algorithm

In [ ]:
WEIGHTS = {
    "author_credibility": 0.25,
    "citation_count":     0.20,
    "domain_authority":   0.25,
    "recency":            0.20,
    "medical_disclaimer": 0.10,
}

DOMAIN_AUTHORITY = {
    "pubmed.ncbi.nlm.nih.gov": 100, "nih.gov": 98,  "who.int": 97,
    "cdc.gov": 96,  "nature.com": 94,  "nejm.org": 93, "thelancet.com": 92,
    "sciencedirect.com": 90, "springer.com": 88, "bmj.com": 87,
    "towardsdatascience.com": 72, "medium.com": 70, "realpython.com": 75,
    "dataquest.io": 68, "kaggle.com": 74, "github.com": 80,
    "stackoverflow.com": 84, "youtube.com": 82, "blogspot.com": 30,
    "wordpress.com": 40,
}

CREDIBLE_ORGS = {"google","stanford","mit","harvard","oxford","cambridge",
                 "microsoft","meta","openai","anthropic","deepmind","nih",
                 "cdc","who","ieee","acm","nature","pubmed","ncbi","freecodecamp"}

SPAM_SIGNALS = ["click here now","buy now","limited offer","100% guaranteed",
                "make money fast","miracle cure","secret revealed"]

def calculate_trust_score(source_type, author, published_date, domain,
                          citation_count=0, has_medical_disclaimer=False, content=""):
    ac  = _author_score(author, source_type)
    cc  = _citation_score(citation_count, source_type)
    da  = _da_score(domain)
    rec = _recency_score(published_date)
    med = 1.0 if (has_medical_disclaimer or source_type=="pubmed") else 0.0

    raw = (WEIGHTS["author_credibility"] * ac +
           WEIGHTS["citation_count"]     * cc +
           WEIGHTS["domain_authority"]   * da +
           WEIGHTS["recency"]            * rec +
           WEIGHTS["medical_disclaimer"] * med)

    penalty = _penalties(author, domain, content, source_type)
    return round(max(0.0, min(1.0, raw - penalty)), 4)

def _author_score(author, source_type):
    if not author or author.strip().lower() in ("unknown","n/a",""):
        return 0.2
    names = [n.strip() for n in author.split(",")]
    scores = []
    for name in names:
        nl = name.lower()
        if any(org in nl for org in CREDIBLE_ORGS): scores.append(0.90)
        elif len(name.split()) >= 2:                scores.append(0.55)
        else:                                        scores.append(0.30)
    score = sum(scores)/len(scores)
    return max(score, 0.70) if source_type == "pubmed" else score

def _citation_score(n, source_type):
    if source_type in ("blog","youtube") and n == 0: return 0.1
    if n <= 0: return 0.05
    return round(min(math.log10(n+1)/math.log10(1001), 1.0), 4)

def _da_score(domain):
    bare = re.sub(r"https?://","",domain).split("/")[0].lower()
    da = DOMAIN_AUTHORITY.get(bare)
    if da is None:
        for key, val in DOMAIN_AUTHORITY.items():
            if bare.endswith(key): da = val; break
    return round((da if da else 35) / 100, 4)

def _recency_score(date_str):
    m = re.search(r"\b(19[5-9]\d|20[0-2]\d)\b", str(date_str))
    if not m: return 0.3
    age = max(0, datetime.now().year - int(m.group(1)))
    return round(math.exp(-math.log(2)/2 * age), 4)

def _penalties(author, domain, content, source_type):
    p = 0.0
    cl = content.lower()
    if not author or author.strip().lower() in ("unknown","admin","staff","editor","webmaster"):
        p += 0.10
    bare = re.sub(r"https?://","",domain).split("/")[0].lower()
    da = DOMAIN_AUTHORITY.get(bare, 35)
    if source_type == "blog" and da < 30: p += 0.12
    health = ["treatment","cure","diagnosis","medicine","drug","symptom","therapy","vaccine"]
    if any(h in cl for h in health) and source_type == "blog": p += 0.08
    spam = sum(1 for s in SPAM_SIGNALS if s in cl)
    if spam: p += min(spam * 0.05, 0.20)
    if content:
        words = re.findall(r"\b[a-z]{4,}\b", cl)
        if words:
            top = Counter(words).most_common(1)[0][1]
            if top / len(words) > 0.03: p += 0.05
    return round(min(p, 0.50), 4)

def score_label(s):
    return "High" if s>=0.80 else "Medium" if s>=0.55 else "Low" if s>=0.30 else "Very Low"

print(" Trust score module ready")


 Trust score module ready


##  Step 6 — Blog Scraper

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"}

BLOG_URLS = [
    "https://realpython.com/beautiful-soup-web-scraper-python/",
    "https://www.dataquest.io/blog/web-scraping-tutorial-python/",
    "https://towardsdatascience.com/the-art-of-data-scraping-a-comprehensive-guide-bce82c8aba9a",
]

def scrape_blog(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        author = _blog_author(soup)
        date   = _blog_date(soup)
        text   = _blog_text(soup)

        try:
            lang = langdetect.detect(text[:500]) if LANGDETECT_OK and text else "en"
        except:
            lang = "en"

        region = _blog_region(soup)
        tags   = auto_tag(text)
        chunks = chunk_text(text)
        score  = calculate_trust_score("blog", author, date, url,
                     has_medical_disclaimer="disclaimer" in text.lower(), content=text)

        print(f"  Blog scraped: {url[:60]}  trust={score}")
        return {"source_url":url,"source_type":"blog","author":author,
                "published_date":date,"language":lang,"region":region,
                "topic_tags":tags,"trust_score":score,"content_chunks":chunks}
    except Exception as e:
        print(f"   Blog failed: {url[:60]} — {e}")
        return _err(url,"blog",str(e))

def _blog_author(soup):
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            d = json.loads(script.string or "")
            if isinstance(d, list): d = d[0]
            a = d.get("author",{})
            if isinstance(a, dict):  return a.get("name","Unknown")
            if isinstance(a, list):  return ", ".join(x.get("name","") for x in a)
        except: pass
    for attr, val in [("name","author"),("property","article:author")]:
        t = soup.find("meta",{attr:val})
        if t and t.get("content"): return t["content"].strip()
    for sel in [".author",".byline",'[rel="author"]',".post-author"]:
        el = soup.select_one(sel)
        if el: return el.get_text(strip=True)
    return "Unknown"

def _blog_date(soup):
    t = soup.find("time")
    if t: return t.get("datetime") or t.get_text(strip=True)
    for name in ["article:published_time","datePublished","DC.date"]:
        m = soup.find("meta",{"property":name}) or soup.find("meta",{"name":name})
        if m and m.get("content"): return m["content"][:10]
    return "Unknown"

def _blog_text(soup):
    for tag in soup(["script","style","nav","footer","header","aside","form"]):
        tag.decompose()
    el = soup.find("article") or soup.find("main") or soup.find("body")
    return el.get_text(separator="\n", strip=True) if el else ""

def _blog_region(soup):
    for name in ["geo.region","og:locale"]:
        t = soup.find("meta",{"name":name}) or soup.find("meta",{"property":name})
        if t and t.get("content"): return t["content"]
    return "Unknown"

def _err(url, stype, error):
    return {"source_url":url,"source_type":stype,"author":"Unknown",
            "published_date":"Unknown","language":"unknown","region":"Unknown",
            "topic_tags":[],"trust_score":0.0,"content_chunks":[],"error":error}

def scrape_all_blogs():
    print("\n Scraping blog posts...")
    return [scrape_blog(u) for u in BLOG_URLS]

print(" Blog scraper ready")


 Blog scraper ready


##  Step 7 — YouTube Scraper

In [ ]:
YOUTUBE_URLS = [
    "https://www.youtube.com/watch?v=HcqpanDadyQ",
    "https://www.youtube.com/watch?v=XVv6mJpFOb0",
]

def scrape_youtube(url):
    vid = re.search(r"v=([a-zA-Z0-9_-]{11})", url)
    if not vid:
        return _err(url, "youtube", "No video ID")
    video_id = vid.group(1)
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        def meta(prop):
            t = soup.find("meta",{"property":prop}) or soup.find("meta",{"name":prop})
            return t["content"].strip() if t and t.get("content") else ""

        channel = meta("og:title") or "Unknown"
        # Try to get real channel name
        t = soup.find("span", itemprop="author")
        if t:
            link = t.find("link", itemprop="name")
            if link: channel = link.get("content", channel)

        date = ""
        for prop in ["datePublished","uploadDate"]:
            t = soup.find("meta",{"itemprop":prop})
            if t and t.get("content"): date = t["content"][:10]; break
        if not date: date = "Unknown"

        desc = meta("og:description") or meta("description")

        transcript = "[Transcript unavailable]"
        try:
            from youtube_transcript_api import YouTubeTranscriptApi
            snips = YouTubeTranscriptApi.get_transcript(video_id)
            transcript = " ".join(s["text"] for s in snips)
        except: pass

        full = f"{channel}\n{desc}\n{transcript}"
        tags   = auto_tag(full)
        chunks = chunk_text(full)
        score  = calculate_trust_score("youtube", channel, date, "youtube.com", content=full)

        print(f"  YouTube scraped: {url[:60]}  trust={score}")
        return {"source_url":url,"source_type":"youtube","author":channel,
                "published_date":date,"language":"en","region":"Unknown",
                "topic_tags":tags,"trust_score":score,"content_chunks":chunks}
    except Exception as e:
        print(f"  YouTube failed: {url[:60]} — {e}")
        return _err(url, "youtube", str(e))

def scrape_all_youtube():
    print("\n Scraping YouTube videos...")
    return [scrape_youtube(u) for u in YOUTUBE_URLS]

print(" YouTube scraper ready")


 YouTube scraper ready


##  Step 8 — PubMed Scraper

In [ ]:
PUBMED_URL = "https://pubmed.ncbi.nlm.nih.gov/37250956/"
EFETCH = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
ELINK  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi"

def scrape_pubmed(url=PUBMED_URL):
    m = re.search(r"/(\d{7,9})/?$", url)
    if not m: return _err(url, "pubmed", "Could not extract PMID")
    pmid = m.group(1)
    try:
        resp = requests.get(EFETCH, params={"db":"pubmed","id":pmid,"rettype":"xml","retmode":"xml"}, timeout=20)
        resp.raise_for_status()
        root = ET.fromstring(resp.content)
        art  = root.find(".//PubmedArticle")
        if art is None: return _err(url, "pubmed", "No article in XML")

        title_el = art.find(".//ArticleTitle")
        title = "".join(title_el.itertext()).strip() if title_el is not None else "Unknown"

        authors = []
        for a in art.findall(".//Author"):
            last = a.findtext("LastName",""); fore = a.findtext("ForeName","")
            coll = a.findtext("CollectiveName","")
            name = f"{fore} {last}".strip() if (fore or last) else coll
            if name: authors.append(name)
        author_str = ", ".join(authors) if authors else "Unknown"

        journal = art.findtext(".//Journal/Title","Unknown")
        year    = art.findtext(".//PubDate/Year","Unknown")
        month   = art.findtext(".//PubDate/Month","")
        date    = f"{year}-{month}" if month else year
        lang_el = art.find(".//Language")
        lang    = lang_el.text.lower() if lang_el is not None else "en"

        abstract_parts = []
        for ab in art.findall(".//AbstractText"):
            label = ab.get("Label","")
            text  = "".join(ab.itertext()).strip()
            abstract_parts.append(f"{label}: {text}" if label else text)
        abstract = "\n".join(abstract_parts) or "No abstract available"

        mesh = [m.findtext("DescriptorName","") for m in art.findall(".//MeshHeading")]
        mesh_tags = [t for t in mesh if t]
        topic_tags = list(dict.fromkeys(mesh_tags + auto_tag(abstract)))[:10]

        citations = _pubmed_citations(pmid)
        full_text = f"Title: {title}\nJournal: {journal}\nAuthors: {author_str}\n\n{abstract}"
        chunks = chunk_text(full_text)
        score  = calculate_trust_score("pubmed", author_str, str(year),
                     "pubmed.ncbi.nlm.nih.gov", citation_count=citations,
                     has_medical_disclaimer=True, content=abstract)

        print(f"  PubMed scraped: PMID {pmid}  trust={score}")
        return {"source_url":url,"source_type":"pubmed","author":author_str,
                "published_date":date,"language":lang,"region":"Unknown",
                "topic_tags":topic_tags,"trust_score":score,"content_chunks":chunks,
                "journal":journal,"title":title}
    except Exception as e:
        print(f"   PubMed failed: {e}")
        return _err(url, "pubmed", str(e))

def _pubmed_citations(pmid):
    try:
        resp = requests.get(ELINK, params={"dbfrom":"pubmed","db":"pubmed","id":pmid,"cmd":"neighbor","retmode":"json"}, timeout=10)
        links = resp.json().get("linksets",[{}])[0].get("linksetdbs",[])
        for l in links:
            if l.get("linkname") == "pubmed_pubmed_citedin":
                return len(l.get("links",[]))
        return 0
    except: return 0

print(" PubMed scraper ready")


 PubMed scraper ready


##  Step 9 — Run Full Pipeline

In [ ]:
print("=" * 60)
print("   RUNNING MULTI-SOURCE SCRAPER PIPELINE")
print("=" * 60)

blogs   = scrape_all_blogs()
yt      = scrape_all_youtube()
pubmed  = [scrape_pubmed()]

all_data = blogs + yt + pubmed

print(f"\n Total records collected: {len(all_data)}")
print("=" * 60)


   RUNNING MULTI-SOURCE SCRAPER PIPELINE

 Scraping blog posts...
  Blog scraped: https://realpython.com/beautiful-soup-web-scraper-python/  trust=0.325
  Blog scraped: https://www.dataquest.io/blog/web-scraping-tutorial-python/  trust=0.165
   Blog failed: https://towardsdatascience.com/the-art-of-data-scraping-a-co — 404 Client Error: Not Found for url: https://towardsdatascience.com/the-art-of-data-scraping-a-comprehensive-guide-bce82c8aba9a

 Scraping YouTube videos...
  YouTube scraped: https://www.youtube.com/watch?v=HcqpanDadyQ  trust=0.4088
  YouTube scraped: https://www.youtube.com/watch?v=XVv6mJpFOb0  trust=0.425
  PubMed scraped: PMID 37250956  trust=0.5557

 Total records collected: 6


##  Step 10 — Save Output JSON Files

In [ ]:
import os
os.makedirs("scraped_data", exist_ok=True)

def save(data, filename):
    path = f"scraped_data/{filename}"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"   Saved: {path}  ({len(data)} records)")

save(blogs,    "blogs.json")
save(yt,       "youtube.json")
save(pubmed,   "pubmed.json")
save(all_data, "scraped_data.json")


   Saved: scraped_data/blogs.json  (3 records)
   Saved: scraped_data/youtube.json  (2 records)
   Saved: scraped_data/pubmed.json  (1 records)
   Saved: scraped_data/scraped_data.json  (6 records)


##  Step 11 — View Results Summary

In [ ]:
print(f"{'Source':<12} {'Type':<10} {'Trust':>6}  {'Label':<10}  Tags")
print("-" * 80)
for rec in all_data:
    author = rec["author"][:20] if rec.get("author") else "Unknown"
    tags   = ", ".join(rec["topic_tags"][:3]) if rec.get("topic_tags") else "—"
    label  = score_label(rec["trust_score"])
    print(f"{author:<22} {rec['source_type']:<10} {rec['trust_score']:>6}  {label:<10}  {tags}")


Source       Type        Trust  Label       Tags
--------------------------------------------------------------------------------
Martin Breuss          blog        0.325  Low         Machine Learning, Deep Learning, Artificial Intelligence
Unknown                blog        0.165  Very Low    Machine Learning, Artificial Intelligence, Data Scraping
Unknown                blog          0.0  Very Low    —
Google Cloud Tech      youtube    0.4088  Low         Machine Learning, Artificial Intelligence, Education
freeCodeCamp.org       youtube     0.425  Low         Artificial Intelligence, Data Scraping, Python
Carter M Armstrong,    pubmed     0.5557  Medium      Healthcare, Vacuum, Application


##  Step 12 — Inspect a Single Record (PubMed)

In [ ]:
import pprint
pprint.pprint(pubmed[0], width=100)


{'author': 'Carter M Armstrong, Emma C Snively, Muhammad Shumail, Christopher Nantista, Zenghai '
           'Li, Sami Tantawi, Bill W Loo, Richard J Temkin, Robert G Griffin, Jinjun Feng, Roberto '
           'Dionisio, Felix Mentgen, Natanael Ayllon, Mark A Henderson, Timothy P Goodman',
 'content_chunks': ['Title: Frontiers in the Application of RF Vacuum Electronics.\n'
                    'Journal: IEEE transactions on electron devices\n'
                    'Authors: Carter M Armstrong, Emma C Snively, Muhammad Shumail, Christopher '
                    'Nantista, Zenghai Li, Sami Tantawi, Bill W Loo, Richard J Temkin, Robert G '
                    'Griffin, Jinjun Feng, Roberto Dionisio, Felix Mentgen, Natanael Ayllon, Mark '
                    'A Henderson, Timothy P Goodman',
                    'The application of radio frequency (RF) vacuum electronics for the betterment '
                    'of the human condition began soon after the invention of the first vacuum '
    

##  Step 13 — Download Files from Colab

In [ ]:
from google.colab import files

for fname in ["blogs.json", "youtube.json", "pubmed.json", "scraped_data.json"]:
    files.download(f"scraped_data/{fname}")
    print(f" Downloading {fname}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>